In [18]:
import zipfile
import os
import rasterio
import numpy as np
import matplotlib.pyplot as plt

In [19]:
zip_path = 'drive-download-20260804T191934Z-1-001.zip'
extract_path = '/content/dataset'

In [20]:
with zipfile.ZipFile(zip_path, "r") as zip_ref:
  zip_ref.extractall(extract_path)
  print("Dataset extracted successfully to:", extract_path)

Dataset extracted successfully to: /content/dataset


In [21]:
os.makedirs(extract_path, exist_ok = True)
os.makedirs("/content/model_ready_dataset", exist_ok=True)
os.makedirs("/content/visuals", exist_ok=True)

In [22]:
def enhance_visualization(band, percent=2):
  valid_mask = np.isfinite(band)
  if not np.any(valid_mask):
    return np.zeros_like(band, dtype=np.float32)
  low, high = np.percentile(band[valid_mask], [percent, 100-percent])
  if high <= low:
    return np.zeros_like(band, dtype=np.float32)

  stretched = np.zeros_like(band, dtype=np.float32)
  stretched[valid_mask] = np.clip((band[valid_mask] - low) / (high - low),0, 1)
  return stretched

In [23]:
def create_image(data):
  rgb_bands = [0, 1, 2]
  h, w = data.shape[1], data.shape[2]
  vis_image = np.zeros((h, w, 3))
  for i, band_idx in enumerate(rgb_bands):
      vis_image[:, :, i] = enhance_visualization(data[band_idx])
  return vis_image

In [24]:
def preprocess(file_path, file_name):
  with rasterio.open(file_path) as src:
    raster_data = src.read()
  cleaned_data = np.copy(raster_data)
  for i in range(cleaned_data.shape[0]):
    band = cleaned_data[i]
    band_mean = np.nanmean(band)
    nan_mask = np.isnan(band)
    band[nan_mask] = band_mean
    print(f"Band {i+1}: Replaced {nan_mask.sum()}")

  normalised_data = np.copy(cleaned_data)

  for i in range(normalised_data.shape[0]):
    band = normalised_data[i]
    p1, p99 = np.percentile(band, [1, 99])
    band_clipped = np.clip(band, p1, p99)
    band_min, band_max = band_clipped.min(), band_clipped.max()
    if band_max > band_min:
        normalised_data[i] = (band_clipped - band_min) / (band_max - band_min)
    else:
        normalised_data[i] = band_clipped - band_min
    print(f"Band {i+1}: Clipped to [{p1:.4f}, {p99:.4f}] and normalized to [0, 1]")

    output_normalised_path = f'/content/model_ready_dataset/{file_name}.tif'
  with rasterio.open(file_path) as src:
    profile = src.profile
    profile.update(dtype = rasterio.float32, nodata = None, count = 8)

  with rasterio.open(output_normalised_path,"w", **profile) as dst:
    dst.write(normalised_data.astype(rasterio.float32))

  print(f"Model-ready data exported to: {output_normalised_path}")

  image = create_image(cleaned_data)
  plt.imsave(f"/content/visuals/{file_name}.png", image)

In [25]:
files = [
    "Sentinel2_Features_Aug12"
    , "Sentinel2_Features_Sep02"
    , "Sentinel2_Features_Sep08"
    , "Sentinel2_Features_Sep16"
    , "Sentinel2_Features_Sep19"
]

In [26]:
for i in files:
  file_path = os.path.join(extract_path, f"{i}.tif")
  preprocess(file_path, i )

Band 1: Replaced 2458858
Band 2: Replaced 2458858
Band 3: Replaced 2458858
Band 4: Replaced 2458858
Band 5: Replaced 2458858
Band 6: Replaced 2458858
Band 7: Replaced 2458858
Band 8: Replaced 2458858
Band 1: Clipped to [0.0098, 0.2796] and normalized to [0, 1]
Band 2: Clipped to [0.0311, 0.3172] and normalized to [0, 1]
Band 3: Clipped to [0.0172, 0.3550] and normalized to [0, 1]
Band 4: Clipped to [0.0850, 0.5058] and normalized to [0, 1]
Band 5: Clipped to [0.1146, 0.4905] and normalized to [0, 1]
Band 6: Clipped to [0.0585, 0.4515] and normalized to [0, 1]
Band 7: Clipped to [-0.0117, 0.9000] and normalized to [0, 1]
Band 8: Clipped to [-0.4364, 0.2945] and normalized to [0, 1]
Model-ready data exported to: /content/model_ready_dataset/Sentinel2_Features_Aug12.tif
Band 1: Replaced 2458858
Band 2: Replaced 2458858
Band 3: Replaced 2458858
Band 4: Replaced 2458858
Band 5: Replaced 2458858
Band 6: Replaced 2458858
Band 7: Replaced 2458858
Band 8: Replaced 2458858
Band 1: Clipped to [0.